# 02_eda_visualization.ipynb

Converted from your cell-by-cell rebuild guide.

## Cell 1 — Imports & load

In [21]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

price   = pd.read_csv("price_clean.csv", parse_dates=["date"])
twitter = pd.read_csv("twitter_clean.csv", parse_dates=["date"])

COINS = ["Bitcoin","Ethereum","Dogecoin"]
COLORS = {"Bitcoin":"#F7931A","Ethereum":"#627EEA","Dogecoin":"#C2A633"}

## Cell 2 — Time-series price plot (all 3 coins)

In [22]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
for ax, coin in zip(axes, COINS):
    g = price[price["coin"]==coin].sort_values("date")
    ax.plot(g["date"], g["close"], color=COLORS[coin], linewidth=1.2, label=coin)
    ax.set_title(f"{coin} — Daily Close Price", fontsize=12, fontweight="bold")
    ax.set_ylabel("USD")
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
plt.suptitle("Daily Close Prices: 2017–2021", fontsize=14)
plt.tight_layout()
plt.savefig("eda_timeseries_price.png", dpi=150)
plt.show()

## Cell 3 — Distribution of daily returns

In [23]:
price_sorted = price.sort_values(["coin","date"]).copy()
price_sorted["daily_return"] = price_sorted.groupby("coin")["close"].pct_change()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, coin in zip(axes, COINS):
    g = price_sorted[price_sorted["coin"]==coin]["daily_return"].dropna()
    ax.hist(g, bins=60, color=COLORS[coin], edgecolor="white", alpha=0.85)
    ax.set_title(f"{coin} Daily Return Distribution")
    ax.set_xlabel("Return")
    ax.set_ylabel("Frequency")
    ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
    ax.text(0.7, 0.9, f"Skew={g.skew():.2f}\nKurt={g.kurtosis():.2f}",
            transform=ax.transAxes, fontsize=9)
plt.suptitle("Daily Return Distributions — Fat Tails & Skew", fontsize=13)
plt.tight_layout()
plt.savefig("eda_return_distributions.png", dpi=150)
plt.show()

## Cell 4 — Up vs Down class comparison (box plots)

In [24]:
price_sorted["target"] = (
    price_sorted.groupby("coin")["close"].shift(-1) > price_sorted["close"]
).astype("Int64")
df = price_sorted.dropna(subset=["target","daily_return"]).copy()
df["Direction"] = df["target"].map({1:"Up ↑", 0:"Down ↓"})

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, coin in zip(axes, COINS):
    g = df[df["coin"]==coin]
    sns.boxplot(data=g, x="Direction", y="daily_return", ax=ax,
                palette={"Up ↑":"#22c55e","Down ↓":"#ef4444"})
    ax.set_title(f"{coin}\nReturn by Next-Day Direction")
    ax.set_xlabel("")
    ax.set_ylabel("Same-day Return")
plt.suptitle("Up vs Down — Same-day Return Distributions", fontsize=13)
plt.tight_layout()
plt.savefig("eda_up_vs_down_boxplot.png", dpi=150)
plt.show()

/tmp/ipykernel_32384/2841172064.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=g, x="Direction", y="daily_return", ax=ax,
/tmp/ipykernel_32384/2841172064.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=g, x="Direction", y="daily_return", ax=ax,
/tmp/ipykernel_32384/2841172064.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=g, x="Direction", y="daily_return", ax=ax,


## Cell 5 — Volume anomalies & spikes

In [25]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, coin in zip(axes, COINS):
    g = price_sorted[price_sorted["coin"]==coin].sort_values("date")
    ax.bar(g["date"], g["volume"], color=COLORS[coin], alpha=0.7, width=1)
    # Mark top 1% volume spikes
    spike_thresh = g["volume"].quantile(0.99)
    spikes = g[g["volume"] >= spike_thresh]
    ax.scatter(spikes["date"], spikes["volume"], color="red", s=20, zorder=5, label="Top 1% spike")
    ax.set_title(f"{coin} — Daily Volume + Spikes")
    ax.set_ylabel("Volume")
    ax.legend()
    ax.grid(alpha=0.2)
plt.suptitle("Transaction Volume Anomalies", fontsize=14)
plt.tight_layout()
plt.savefig("eda_volume_spikes.png", dpi=150)
plt.show()

## Cell 6 — Temporal patterns (day of week & month)

In [26]:
df["dayofweek"] = df["date"].dt.dayofweek
df["month"]     = df["date"].dt.month

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
day_labels = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
for i, coin in enumerate(COINS):
    g = df[df["coin"]==coin]
    # Up rate by day of week
    dow_rate = g.groupby("dayofweek")["target"].mean()
    axes[0,i].bar(dow_rate.index, dow_rate.values, color=COLORS[coin], alpha=0.85)
    axes[0,i].set_xticks(range(7)); axes[0,i].set_xticklabels(day_labels, fontsize=9)
    axes[0,i].set_title(f"{coin}\nUp% by Day of Week"); axes[0,i].set_ylim(0,1)
    axes[0,i].axhline(0.5, color="gray", linestyle="--")

    # Up rate by month
    mon_rate = g.groupby("month")["target"].mean()
    axes[1,i].bar(mon_rate.index, mon_rate.values, color=COLORS[coin], alpha=0.85)
    axes[1,i].set_title(f"{coin}\nUp% by Month"); axes[1,i].set_ylim(0,1)
    axes[1,i].axhline(0.5, color="gray", linestyle="--")

plt.suptitle("Temporal Patterns — Day of Week & Month", fontsize=14)
plt.tight_layout()
plt.savefig("eda_temporal_patterns.png", dpi=150)
plt.show()

## Cell 7 — Correlation heatmap per coin

In [27]:
feature_cols_corr = ["open","high","low","close","volume","daily_return"]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, coin in zip(axes, COINS):
    g = df[df["coin"]==coin][feature_cols_corr + ["target"]].dropna()
    corr = g.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
                ax=ax, vmin=-1, vmax=1, linewidths=0.5, cbar_kws={"shrink":0.8})
    ax.set_title(f"{coin} Correlation Heatmap")
plt.suptitle("Feature Correlation with Target (daily direction)", fontsize=13)
plt.tight_layout()
plt.savefig("eda_correlation_heatmap.png", dpi=150)
plt.show()

## Cell 8 — Twitter sentiment over time

In [28]:
# Only run if twitter data available
if len(twitter) > 0:
    fig, axes = plt.subplots(3, 1, figsize=(14, 9))
    for ax, coin in zip(axes, COINS):
        g = twitter[twitter["coin"]==coin].sort_values("date")
        if len(g) == 0:
            ax.set_title(f"{coin} — no Twitter data"); continue
        ax.plot(g["date"], g["sentiment_score"], color=COLORS[coin], alpha=0.7)
        ax.fill_between(g["date"], g["sentiment_score"], 0,
                        where=(g["sentiment_score"]>0), alpha=0.2, color="green", label="Positive")
        ax.fill_between(g["date"], g["sentiment_score"], 0,
                        where=(g["sentiment_score"]<=0), alpha=0.2, color="red", label="Negative")
        ax.axhline(0, color="gray", linestyle="--")
        ax.set_title(f"{coin} Twitter Sentiment Over Time")
        ax.set_ylabel("Sentiment Score")
        ax.legend()
    plt.suptitle("Twitter Sentiment: 2017–2021 Overlap Period", fontsize=14)
    plt.tight_layout()
    plt.savefig("eda_twitter_sentiment_timeseries.png", dpi=150)
    plt.show()

## Cell 9 — Class imbalance bar chart

In [29]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, coin in zip(axes, COINS):
    g = df[df["coin"]==coin]["target"].value_counts().sort_index()
    bars = ax.bar(["Down ↓","Up ↑"], g.values, color=["#ef4444","#22c55e"], edgecolor="white")
    for bar, val in zip(bars, g.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val),
                ha="center", va="bottom", fontsize=10)
    ax.set_title(f"{coin}\nClass Distribution")
    ax.set_ylabel("Count")
plt.suptitle("Class Imbalance Check — Up vs Down Days", fontsize=13)
plt.tight_layout()
plt.savefig("eda_class_imbalance.png", dpi=150)
plt.show()
print("💡 Note: class imbalance will be handled via SMOTE + class_weight in modelling notebooks.")

/tmp/ipykernel_32384/2152989976.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, 3, figsize=(12, 4))


💡 Note: class imbalance will be handled via SMOTE + class_weight in modelling notebooks.


## Cell 10 — Bias & risk analysis summary

In [30]:
# Potential biases identified:
biases = {
    "Survivorship Bias": "Only coins still active in 2021 — Dogecoin started 2017",
    "Temporal Non-stationarity": "Bull/bear cycles cause concept drift",
    "Twitter Sentiment Coverage": "Twitter data may favour larger coins (BTC/ETH)",
    "Thin Market": "Dogecoin had very low volume before 2020 — features less stable",
    "Class Imbalance": "Markets trend ~52–55% up on average — slight imbalance",
    "Lookahead in Labels": "Target is strictly t+1 close vs t close — no leakage",
}
print("=== Identified Biases & Risks ===")
for k, v in biases.items():
    print(f"\n⚠️  {k}\n   → {v}")

=== Identified Biases & Risks ===

⚠️  Survivorship Bias
   → Only coins still active in 2021 — Dogecoin started 2017

⚠️  Temporal Non-stationarity
   → Bull/bear cycles cause concept drift

⚠️  Twitter Sentiment Coverage
   → Twitter data may favour larger coins (BTC/ETH)

⚠️  Thin Market
   → Dogecoin had very low volume before 2020 — features less stable

⚠️  Class Imbalance
   → Markets trend ~52–55% up on average — slight imbalance

⚠️  Lookahead in Labels
   → Target is strictly t+1 close vs t close — no leakage
